¡Por fin tenemos a los tres competidores en la línea de meta! Has completado con absoluto éxito el entrenamiento de todos tus algoritmos. 

Los resultados de **XGBoost** que me acabas de enviar son fascinantes, especialmente cuando los contrastamos con el comportamiento de Random Forest. Aquí ocurre un fenómeno clásico del Machine Learning avanzado: la diferencia entre la estrategia de *Bagging* (Random Forest) y *Boosting* (XGBoost).

Aquí tienes el análisis estructurado y redactado en formato Markdown, listo para tu memoria de título.

***

### Análisis de Resultados: Optimización y Predicción mediante XGBoost

En esta sección se presentan los resultados obtenidos al aplicar el algoritmo de *Gradient Boosting* (XGBoost) para la predicción de los tres desenlaces clínicos y operativos. Se destaca el uso del parámetro computacional `tree_method='hist'`, el cual permitió procesar la totalidad de la matriz de datos en una fracción del tiempo requerido por otros ensambles.

#### 1. Eficiencia Computacional y Estabilidad del Modelo
El contraste en los tiempos de entrenamiento es uno de los hallazgos técnicos más notables de esta fase. Mientras que la búsqueda de hiperparámetros en Random Forest requirió un promedio de 5 horas por variable, XGBoost completó la validación cruzada estratificada (con 5 pliegues) en tiempos extraordinariamente reducidos:
*   **Mortalidad:** 4.05 minutos.
*   **Severidad:** 14.89 minutos.
*   **Consumo de Recursos:** 10.74 minutos.

Para las tres variables, la configuración empírica óptima y más estable convergió en los límites superiores del espacio de búsqueda (`learning_rate`: 0.3, `max_depth`: 10), demostrando desviaciones estándar mínimas (std $\le$ 0.0011). Esto indica que el algoritmo logró asimilar la complejidad de los datos GRD de manera consistente y sin sobreajuste (*overfitting*) atribuible a la partición de la muestra.

#### 2. Evaluación Predictiva (Conjunto de Prueba 2024)

El comportamiento predictivo de XGBoost presentó una dualidad interesante dependiendo de la dimensionalidad de la variable objetivo:

**A) Mortalidad (Clasificación Binaria con Desbalance Extremo):**
*   **F1-Score Macro:** Alcanzó **0.6227**, superando a la Regresión Logística (0.57) y cumpliendo el umbral mínimo (0.60), pero quedando por debajo del Random Forest (0.68).
*   **Dinámica Recall-Precisión:** Debido a la inyección matemática del parámetro `scale_pos_weight` (calculado automáticamente en 21.73 para priorizar la clase minoritaria), el modelo se volvió hiper-sensible. Logró un **Recall del 81%** en pacientes fallecidos (detectó a casi todos), pero a costa de una baja Precisión (18%). Generó un volumen significativo de "falsas alarmas", lo que penalizó su F1-Score en la clase minoritaria (0.29).
*   **AUPRC y Calibración:** Logró un AUPRC robusto de 0.3871 y un Brier Score altamente calibrado de 0.0628.

**B) Severidad y Consumo de Recursos (Multiclase):**
En escenarios multiclase, los mecanismos internos de XGBoost demostraron una ligera superioridad frente al resto de los algoritmos:
*   **Severidad:** Obtuvo un **F1-Macro de 0.7221** (ligeramente superior al 0.7176 de Random Forest), con un excelente AUC-ROC de 0.9034 y un Brier Score de 0.0902.
*   **Consumo de Recursos:** Registró un **F1-Macro de 0.7322** (superando el 0.7287 de Random Forest), destacando su Área Bajo la Curva Precisión-Recall (AUPRC) de 0.8205 y una calibración de 0.1191.

#### 3. Análisis de Importancia de Variables: El Enfoque *Greedy* vs Global
La extracción de las *Feature Importances* revela la naturaleza del algoritmo XGBoost (*Boosting*), el cual utiliza una heurística *greedy* (voraz) que prioriza reglas de división inmediatas y drásticas, a diferencia del enfoque global y suavizado del Random Forest:

1.  **Mortalidad (Dominancia del Ingreso):** El modelo determinó que ingresar por urgencia (**TIPO_INGRESO_URGENCIA**) captura el **41.4%** de la importancia relativa para predecir la muerte. A diferencia del Random Forest (que repartió la importancia entre días de estadía y número de procedimientos), XGBoost construyó sus árboles basándose casi enteramente en el estado crítico de ingreso del paciente.
2.  **Severidad (Segmentación Diagnóstica):** Para clasificar los 4 niveles de gravedad, el modelo otorgó el mayor peso (7.7%) a la categoría **CATEGORIA_CANCER_C81_C96** (neoplasias de tejidos linfoides y hematopoyéticos). El algoritmo identificó que este clúster diagnóstico específico es el principal divisor natural de la gravedad del episodio.
3.  **Consumo de Recursos (Eventos Operativos):** Para separar los terciles de gasto, la heurística detectó que el **TIPO_INGRESO_OBSTETRICO** concentra un asombroso **24.5%** de la importancia. Esto es clínicamente coherente con el sistema GRD, ya que las vías de manejo obstétrico (incluso en pacientes con comorbilidades oncológicas) tienen rutas clínicas y canastas de recursos presupuestarios rígidamente estructuradas.

**Conclusión Parcial:**
XGBoost demuestra ser el algoritmo más eficiente y preciso para predecir desenlaces operativos multiclase (Severidad y Consumo) en los datos GRD. Sin embargo, para la predicción del desenlace vital (Mortalidad), Random Forest se mantiene como el modelo preferido al exhibir un mejor equilibrio entre Precisión y Recall, evitando la sobre-penalización de la clase minoritaria.

In [ ]:
# pip install xgboost
import pandas as pd # Importa la librería para manipulación y análisis de datos tabulares en DataFrames
import numpy as np # Importa la librería para cálculos numéricos avanzados y manejo de arreglos
import os # Importa el módulo para interactuar con el sistema operativo (rutas y carpetas)
import gc # Importa el recolector de basura (Garbage Collector) para liberar memoria RAM manualmente
import time # Importa el módulo para registrar y medir el tiempo exacto de ejecución
from sklearn.model_selection import GridSearchCV # Importa el orquestador de búsqueda exhaustiva de hiperparámetros
from sklearn.model_selection import StratifiedKFold # Importa la herramienta para validación cruzada estratificada con semilla fija
from sklearn.base import clone # Importa la función para clonar la arquitectura de un modelo sin los datos entrenados
import xgboost as xgb # Importa la librería extrema de Gradient Boosting (XGBoost)
from sklearn.preprocessing import label_binarize # Importa la función para convertir etiquetas multiclase en matrices binarias (One-vs-Rest)
from sklearn.metrics import ( # Importa el bloque de herramientas para evaluar el rendimiento
    f1_score, # Métrica que evalúa el balance entre precisión y recall
    average_precision_score, # Métrica para calcular el Área Bajo la Curva Precision-Recall (AUPRC)
    roc_auc_score, # Métrica para calcular el Área Bajo la Curva ROC (AUC-ROC)
    brier_score_loss, # Métrica para evaluar la exactitud y calibración de las probabilidades
    classification_report # Genera un reporte detallado con las métricas por cada clase
)

def entrenar_evaluar_xgb(target_name):
    """
    - Descripción: Función que entrena, optimiza y evalúa un modelo XGBoost.
                   Calcula pesos automáticos para clases desbalanceadas (si es binario),
                   aplica validación cruzada con semilla fija, filtra configuraciones inestables
                   (desviación estándar > 0.10) y extrae las importancias relativas de las variables.
    - Entrada: target_name (str) - Nombre de la columna objetivo a predecir (ej. 'MORTALIDAD', 'SEVERIDAD').
    - Salida: Genera archivos CSV físicos con el historial de GridSearch, métricas del modelo y Feature Importances.
    """
    # 1. Configuración inicial
    dir_datos = "../../Datos/Datasets Finales" # Ruta relativa a los archivos CSV de entrada
    dir_resultados = "../../Resultados/Resultados (etapa 3)/XGBoost" # Ruta donde se guardarán los resultados
    os.makedirs(dir_resultados, exist_ok=True) # Crea el directorio de destino de forma segura si no existe

    # Variables excluidas en el entrenamiento (no se usan como predictores)
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER'] 

    print("="*60)
    print(f"INICIANDO PIPELINE XGBOOST PARA: {target_name.upper()}") # Imprime la variable objetivo a procesar
    print("="*60) 

    # 2. Cargar datos de entrenamiento
    # Etapa 1: Cargar los datasets de entrenamiento (2019-2023) para ambos grupos (oncológico y control)
    print("[1/5] Cargando datasets de entrenamiento...") # Informa progreso de carga de datos
    df_onco_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_onco_2019_2023.csv"), low_memory=False) # Para entrenamiento oncológico
    df_control_train = pd.read_csv(os.path.join(dir_datos, "dataset_entrenamiento_control_2019_2023.csv"), low_memory=False) # Para entrenamiento de control

    # 3. Crear Dataset maestro balanceado (oncológico vs control)
    # Etapa 2: Generar un dataset de entrenamiento balanceado para evitar sesgos en el modelo (igual cantidad de oncológicos y controles)
    print("[2/5] Generando muestra balanceada...") # Informa progreso del balanceo base
    n_onco = len(df_onco_train) # Extrae el total de filas del grupo oncológico
    df_control_sample = df_control_train.sample(n=n_onco, random_state=42) # Extrae aleatoriamente controles para igualar la cantidad (semilla 42)
    df_train_maestro = pd.concat([df_onco_train, df_control_sample], ignore_index=True) # Apila ambos grupos en el dataframe definitivo

    del df_onco_train, df_control_train, df_control_sample # Borra las variables temporales masivas
    gc.collect() # Limpia la RAM inmediatamente

    # Separar X e Y de entrenamiento
    features = [col for col in df_train_maestro.columns if col not in cols_excluir] # Filtra reteniendo solo las variables predictoras válidas
    X_train = df_train_maestro[features] # Define la matriz de atributos independientes (X)
    y_train = df_train_maestro[target_name] # Define el vector objetivo a predecir (Y)
    
    clases_unicas = np.unique(y_train) # Identifica las clases únicas (ej. [0, 1] o [0, 1, 2, 3])
    is_multiclass = len(clases_unicas) > 2 # Verifica lógicamente si el problema es multiclase

    print(f"      -> Dimensiones: {X_train.shape} | Clases: {len(clases_unicas)} (Multiclase: {is_multiclass})") # Imprime la forma del problema

    # 4. Configurar Grid Search (5 Pliegues) con semilla
    # Etapa 3: Configurar la búsqueda exhaustiva de hiperparámetros con validación cruzada estratificada y semilla fija para reproducibilidad
    print("[3/5] Configurando Grid Search CV (K=5) con semilla...") # Informa progreso de configuración
    
    cv_estrategia = StratifiedKFold(n_splits=5, shuffle=True, random_state=42) # Configura la validación cruzada garantizando siempre los mismos cortes (semilla 42)

    # Cálculo "Auto" de scale_pos_weight solicitado en la tabla de hiperparámetros (Solo aplica a binario)
    if not is_multiclass: # Si la variable objetivo es Mortalidad (binaria)
        conteo_clases = y_train.value_counts() # Cuenta cuántos vivos y fallecidos hay internamente
        peso_clase_positiva = conteo_clases[0] / conteo_clases[1] # Calcula el radio de desbalance (negativos / positivos) para XGBoost
        print(f"      -> scale_pos_weight calculado (Auto): {peso_clase_positiva:.2f}") # Muestra el peso que se inyectará al modelo
        
        xgb_base = xgb.XGBClassifier( # Instancia el modelo XGBoost binario
            tree_method='hist', # Activa el modo histograma ultra rápido para datasets masivos
            scale_pos_weight=peso_clase_positiva, # Asigna el peso calculado para priorizar la clase minoritaria
            random_state=42, # Fija la semilla matemática interna de los árboles
            n_jobs=-1 # Habilita el uso de todos los núcleos del procesador internamente para los árboles
        )
    else: # Si el problema es Severidad o Consumo (multiclase)
        xgb_base = xgb.XGBClassifier( # Instancia el modelo XGBoost multiclase
            tree_method='hist', # Activa el modo histograma
            random_state=42, # Fija la semilla matemática
            n_jobs=-1 # Usa todo el procesador interno para construir los árboles
        )

    # Hiperparámetros a explorar en la cuadrícula (6 combinaciones posibles) 
    espacio_hiperparametros = {
        'learning_rate': [0.01, 0.1, 0.3], # Tasa de aprendizaje o magnitud de corrección de errores
        'max_depth': [3, 6, 10] # Profundidad máxima permitida para cada árbol individual
    }

    grid_search = GridSearchCV( # Instancia la validación cruzada orquestada
        estimator=xgb_base, # Usa el XGBoost base creado arriba
        param_grid=espacio_hiperparametros, # Inyecta las 9 configuraciones posibles
        cv=cv_estrategia, # Asigna los 5 pliegues asegurados mediante StratifiedKFold
        scoring='f1_macro', # Establece F1-Macro como métrica a maximizar
        n_jobs=1, # Evalúa los pliegues secuencialmente de a uno para no agotar la RAM
        verbose=3 # Activa el reporte detallado por pliegue en la consola
    )

    # 5. Entrenar y extraer métricas filtradas
    # Etapa 4: Ejecutar el proceso de entrenamiento y optimización con monitoreo de tiempo y filtrado de configuraciones inestables (std > 0.10)
    print("[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...") # Anuncia inicio del entrenamiento
    inicio = time.time() # Registra el reloj del sistema al empezar
    grid_search.fit(X_train, y_train) # Desata el ajuste matemático de las combinaciones y pliegues
    fin = time.time() # Registra el reloj al terminar
    print(f"      -> Búsqueda completada en {round((fin - inicio)/60, 2)} minutos.") # Muestra el tiempo invertido total

    # -------------------------------------------------------------------------
    # Extracción de resultados, generación de respaldo CSV y filtrado metodológico
    # -------------------------------------------------------------------------
    cv_resultados = pd.DataFrame(grid_search.cv_results_) # Pasa la memoria de resultados a un DataFrame tabular
    
    ruta_cv = os.path.join(dir_resultados, f"Resultados_GridSearch_XGB_{target_name}.csv") # Genera la ruta del archivo de historial
    cv_resultados.to_csv(ruta_cv, index=False) # Guarda el reporte de todas las iteraciones en el disco duro
    print(f"      -> Evidencia de hiperparámetros guardada en: {ruta_cv}")

    config_estables = cv_resultados[cv_resultados['std_test_score'] <= 0.10] # Filtra las opciones inestables (varianza alta) según propuesta

    if config_estables.empty: # En caso extremo de que ninguna configuración pasara la prueba de estabilidad
        print("      ADVERTENCIA: Todas las configuraciones tienen std > 0.10.") # Informa el problema
        print("      Se utilizará la de mayor promedio por defecto.") # Toma la decisión por defecto de Scikit-Learn
        mejor_modelo = grid_search.best_estimator_ # Captura el modelo ganador base
    else: # Si hubieron configuraciones aprobadas (lo esperado)
        mejor_indice = config_estables['mean_test_score'].idxmax() # Busca en qué fila está el F1 promedio más alto de las estables
        mejores_params = config_estables.loc[mejor_indice, 'params'] # Saca el diccionario de hiperparámetros de esa fila
        mejor_f1 = config_estables.loc[mejor_indice, 'mean_test_score'] # Recupera el valor F1 numérico promedio
        mejor_std = config_estables.loc[mejor_indice, 'std_test_score'] # Recupera el valor de la desviación estándar
        
        print(f"      -> Mejor configuración estable encontrada:") # Anuncia éxito
        print(f"         Hiperparámetros: {mejores_params}") # Detalla cuáles parámetros ganaron
        print(f"         F1-Macro Promedio: {mejor_f1:.4f} (std: {mejor_std:.4f})") # Muestra su performance documentada

        mejor_modelo = clone(grid_search.estimator) # Crea una nueva red vacía de XGBoost manteniendo random_state y n_jobs
        mejor_modelo.set_params(**mejores_params) # Le asigna estrictamente los hiperparámetros que ganaron
        mejor_modelo.fit(X_train, y_train) # Lo entrena de forma definitiva con el 100% de la matriz de entrenamiento
        
    # -------------------------------------------------------------------------

    del df_train_maestro, X_train, y_train # Destruye las variables masivas
    gc.collect() # Libera RAM del sistema operativo

    # 6. Evaluación en el Conjunto de Prueba
    # Etapa 5: Evaluación de la capacidad de generalización del modelo sobre el futuro (dataset del año 2024)
    print("[5/5] Evaluando en conjunto de prueba (2024)...") # Inicia fase final contra datos no vistos
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), low_memory=False) # Trae datos de prueba oncológico (2024)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), low_memory=False) # Trae datos de prueba control (2024)

    df_test_maestro = pd.concat([df_onco_test, df_control_test], ignore_index=True) # Fusiona ambos para armar el testeo final
    X_test = df_test_maestro[features] # Oculta las respuestas dejando solo las pistas clínicas
    y_test = df_test_maestro[target_name] # Establece las respuestas oficiales definitivas

    y_pred = mejor_modelo.predict(X_test) # Realiza las decisiones absolutas del modelo (ej. 1 o 0)
    y_pred_proba = mejor_modelo.predict_proba(X_test) # Realiza las decisiones en formato probabilístico continuo

    print("\n--- RESULTADOS FINALES ---") 
    print(classification_report(y_test, y_pred)) # Genera tabla textual estándar con métricas precision, recall y F1 por clase
    
    f1_macro_val = f1_score(y_test, y_pred, average='macro') # Extrae el indicador F1 general ponderado
    
    if is_multiclass: # Lógica para múltiples salidas (Severidad / Consumo)
        y_test_bin = label_binarize(y_test, classes=clases_unicas) # Abre las etiquetas multiclase en múltiples columnas binarias
        auc_roc_val = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted') # Calcula AUC-ROC global ponderado (One-vs-Rest)
        auprc_val = average_precision_score(y_test_bin, y_pred_proba, average='weighted') # Calcula el AUPRC ajustando el desbalance de soporte
        brier_val = np.mean([brier_score_loss(y_test_bin[:, k], y_pred_proba[:, k]) for k in range(len(clases_unicas))]) # Genera puntaje Brier iterando y promediando clases
        
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Imprime resultado F1
        print(f"AUPRC (OvR Weighted): {auprc_val:.4f}") # Imprime resultado AUPRC
        print(f"AUC-ROC (OvR Weighted): {auc_roc_val:.4f}") # Imprime resultado ROC
        print(f"Brier Score (Multiclase): {brier_val:.4f}") # Imprime resultado de calibración Brier
            
    else: # Lógica simple de dos salidas (Mortalidad)
        f1_clase1_val = f1_score(y_test, y_pred, pos_label=1) # Fuerza el análisis sobre la capacidad de detectar a la clase 1 (fallecidos)
        auc_roc_val = roc_auc_score(y_test, y_pred_proba[:, 1]) # Extrae AUC-ROC de la columna probabilística positiva
        auprc_val = average_precision_score(y_test, y_pred_proba[:, 1]) # Extrae AUPRC sobre la clase minoritaria
        brier_val = brier_score_loss(y_test, y_pred_proba[:, 1]) # Evalúa la exactitud de las probabilidades (Brier Score)
        
        print(f"F1-Score (Clase 1): {f1_clase1_val:.4f}") # Imprime F1 del fallecimiento
        print(f"F1-Score (Macro): {f1_macro_val:.4f}") # Imprime F1 global
        print(f"AUPRC: {auprc_val:.4f}") # Imprime AUPRC
        print(f"AUC-ROC: {auc_roc_val:.4f}") # Imprime AUC-ROC
        print(f"Brier Score: {brier_val:.4f}") # Imprime métrica de Brier

    # 7. Extraer Feature Importances (Reemplaza a los Odds Ratio en algoritmos de árbol)
    importancias = mejor_modelo.feature_importances_ # Toma del motor XGBoost el vector que mide la importancia relativa de variables
    
    df_importancias = pd.DataFrame({ # Crea una tabla para leer estos datos estructuradamente
        'Variable': features, # Empareja la importancia con el nombre del atributo original
        'Importancia_Relativa': importancias # Asigna el valor del peso interno
    }).sort_values(by='Importancia_Relativa', ascending=False) # Mueve las variables de mayor impacto al inicio
    
    df_importancias = df_importancias[df_importancias['Importancia_Relativa'] > 0] # Filtra eliminando predictores ignorados por el modelo
    
    ruta_imp = os.path.join(dir_resultados, f"XGB_Feature_Importances_{target_name}.csv") # Dinamiza el nombre de guardado del CSV
    df_importancias.to_csv(ruta_imp, index=False) # Escribe el CSV físicamente
    
    print(f"Importancias de Variables guardadas en: {ruta_imp}") # Confirma guardado exitoso
    
    del df_test_maestro, X_test, y_test # Borra variables del 2024 de memoria RAM
    gc.collect() # Limpia la memoria final
    print("="*60, "\n") 

In [2]:
entrenar_evaluar_xgb('MORTALIDAD')

INICIANDO PIPELINE XGBOOST PARA: MORTALIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 2 (Multiclase: False)
[3/5] Configurando Grid Search CV (K=5) con semilla...
      -> scale_pos_weight calculado (Auto): 21.73
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   5.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   3.8s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.536 total time=   4.0s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.538 total time=   3.7s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.537 total time=   3.6s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.563 total time=   5.1s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.563 total time=   5.2

In [3]:
entrenar_evaluar_xgb('SEVERIDAD')


INICIANDO PIPELINE XGBOOST PARA: SEVERIDAD
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 4 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.663 total time=  15.4s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.664 total time=  13.3s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.666 total time=  13.4s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.664 total time=  13.4s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.665 total time=  13.3s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.703 total time=  20.1s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.703 total time=  20.0s
[CV 3/5] END ...learning_rate=0.01, max_depth=6;,

In [4]:
entrenar_evaluar_xgb('CONSUMO_RECURSOS')

INICIANDO PIPELINE XGBOOST PARA: CONSUMO_RECURSOS
[1/5] Cargando datasets de entrenamiento...
[2/5] Generando muestra balanceada...
      -> Dimensiones: (781264, 110) | Clases: 3 (Multiclase: True)
[3/5] Configurando Grid Search CV (K=5) con semilla...
[4/5] Entrenando modelo y evaluando configuraciones (monitoreo activado)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END ...learning_rate=0.01, max_depth=3;, score=0.552 total time=  11.1s
[CV 2/5] END ...learning_rate=0.01, max_depth=3;, score=0.551 total time=   9.3s
[CV 3/5] END ...learning_rate=0.01, max_depth=3;, score=0.548 total time=   9.3s
[CV 4/5] END ...learning_rate=0.01, max_depth=3;, score=0.549 total time=   9.4s
[CV 5/5] END ...learning_rate=0.01, max_depth=3;, score=0.550 total time=   9.2s
[CV 1/5] END ...learning_rate=0.01, max_depth=6;, score=0.628 total time=  13.8s
[CV 2/5] END ...learning_rate=0.01, max_depth=6;, score=0.627 total time=  13.5s
[CV 3/5] END ...learning_rate=0.01, max_dep

In [1]:
import pandas as pd # Importa Pandas para leer los archivos CSV
import numpy as np # Importa Numpy para cálculos matemáticos
import os # Importa OS para manejar las rutas de los archivos

def validar_auprc_lift_automatico(target_name, auprc_obtenido, is_multiclass=False):
    """
    - Descripción: Lee automáticamente el conjunto de prueba (2024), calcula la tasa base (prevalencia)
                   directamente de los datos y verifica matemáticamente si el AUPRC cumple con Lift > 3.0.
    - Entrada: target_name (str) - Nombre del objetivo, auprc_obtenido (float) - El AUPRC de tu mejor modelo,
               is_multiclass (bool) - True si tiene más de 2 clases.
    - Salida: Imprime el reporte de validación en la consola.
    """
    print("="*60) # Separador visual
    print(f"VALIDACIÓN AUTOMÁTICA DE AUPRC: {target_name.upper()}") # Título
    print("="*60) # Separador visual
    
    # 1. Leer SOLO la columna objetivo de los archivos de 2024 (es ultrarrápido y no gasta RAM)
    dir_datos = "../../Datos/Datasets Finales" # Define la ruta relativa
    df_onco = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco_2024.csv"), usecols=[target_name]) # Carga onco
    df_control = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control_2024.csv"), usecols=[target_name]) # Carga control
    
    # 2. Unir y contar las instancias reales
    y_test = pd.concat([df_onco, df_control], ignore_index=True)[target_name] # Une ambos vectores
    clases, soportes_clases = np.unique(y_test, return_counts=True) # Extrae las clases y cuenta cuántas hay de cada una
    total_instancias = len(y_test) # Obtiene el total de pacientes evaluados
    
    # 3. Calcular la Tasa Base según el tipo de problema
    if not is_multiclass: # Si es Mortalidad (Binario)
        indice_clase_1 = np.where(clases == 1)[0][0] # Encuentra en qué posición del arreglo está la clase 1 (Fallecidos)
        soporte_clase_positiva = soportes_clases[indice_clase_1] # Extrae la cantidad exacta de fallecidos
        tasa_base = soporte_clase_positiva / total_instancias # Calcula la prevalencia pura
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Casos reales clase minoritaria (1): {soporte_clase_positiva}") # Muestra casos positivos
        print(f"Tasa base (Prevalencia): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra porcentaje
        
    else: # Si es Severidad o Consumo (Multiclase)
        prevalencias = [soporte / total_instancias for soporte in soportes_clases] # Calcula la prevalencia individual de cada clase
        tasa_base = sum([p**2 for p in prevalencias]) # Calcula la prevalencia esperada por azar (promedio ponderado OvR)
        
        print(f"Total episodios de prueba: {total_instancias}") # Muestra total
        print(f"Prevalencias exactas por clase: {[round(p, 4) for p in prevalencias]}") # Muestra cómo se reparte la torta
        print(f"Tasa base ponderada (OvR): {tasa_base:.4f} ({tasa_base*100:.2f}%)") # Muestra el azar ponderado

    # 4. Calcular el umbral y el Lift real
    umbral_minimo = tasa_base * 3.0 # Calcula el AUPRC mínimo que exige tu tesis
    lift_real = auprc_obtenido / tasa_base # Calcula cuántas veces mejor que el azar es tu modelo
    
    print("-" * 60) # Separador
    print(f"- AUPRC Obtenido por tu modelo: {auprc_obtenido:.4f}") # Muestra tu resultado
    print(f"- AUPRC Mínimo exigido (Tasa Base x 3.0): {umbral_minimo:.4f}") # Muestra el límite inferior
    print(f"- LIFT REAL LOGRADO: {lift_real:.2f}x") # Muestra el multiplicador de mejora
    
    # 5. Veredicto final
    if auprc_obtenido > umbral_minimo: # Si supera el umbral
        print("RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)")
    else: # Si no lo supera
        print("RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)")
    print("\n")

In [2]:
validar_auprc_lift_automatico(
    target_name='MORTALIDAD', 
    auprc_obtenido=0.3871, 
    is_multiclass=False
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: MORTALIDAD
Total episodios de prueba: 1076345
Casos reales clase minoritaria (1): 26221
Tasa base (Prevalencia): 0.0244 (2.44%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.3871
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.0731
- LIFT REAL LOGRADO: 15.89x
RESULTADO: ¡CUMPLE LA CONDICIÓN ESTRICTA DE LA TESIS! (Lift > 3.0)




In [3]:
validar_auprc_lift_automatico(
    target_name='SEVERIDAD', 
    auprc_obtenido=0.7814, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: SEVERIDAD
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [np.float64(0.1969), np.float64(0.3561), np.float64(0.2478), np.float64(0.1992)]
Tasa base ponderada (OvR): 0.2667 (26.67%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.7814
- AUPRC Mínimo exigido (Tasa Base x 3.0): 0.8000
- LIFT REAL LOGRADO: 2.93x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)




In [4]:
validar_auprc_lift_automatico(
    target_name='CONSUMO_RECURSOS', 
    auprc_obtenido=0.8205, 
    is_multiclass=True
)

VALIDACIÓN AUTOMÁTICA DE AUPRC: CONSUMO_RECURSOS
Total episodios de prueba: 1076345
Prevalencias exactas por clase: [np.float64(0.3345), np.float64(0.3367), np.float64(0.3287)]
Tasa base ponderada (OvR): 0.3334 (33.34%)
------------------------------------------------------------
- AUPRC Obtenido por tu modelo: 0.8205
- AUPRC Mínimo exigido (Tasa Base x 3.0): 1.0001
- LIFT REAL LOGRADO: 2.46x
RESULTADO: NO CUMPLE LA CONDICIÓN (Revisar factibilidad matemática por balanceo de clases)


